# 05 — Weight of Evidence (WoE) & Information Value (IV) Analysis
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Module:** Credit Scorecard WoE/IV Analysis (Module 3 — Charukesh)

## 1. Mathematical Formulation & Leakage Prevention
To prevent target leakage, quantile bin boundaries and category WoE mappings are fitted **strictly on the stratified training set (`n = 22,691`)**. Zero-frequency bins are protected via **Laplace smoothing ($\alpha = 0.5$)**:

$$\text{Dist}_{\text{Event}, i} = \frac{E_i + \alpha}{\sum_j (E_j + \alpha)}, \quad \text{Dist}_{\text{NonEvent}, i} = \frac{NE_i + \alpha}{\sum_j (NE_j + \alpha)}$$

$$\text{WoE}_i = \ln\!\left(\frac{\text{Dist}_{\text{Event}, i}}{\text{Dist}_{\text{NonEvent}, i}}\right), \quad \text{IV} = \sum_{i=1}^{k} \left(\text{Dist}_{\text{Event}, i} - \text{Dist}_{\text{NonEvent}, i}\right) \cdot \text{WoE}_i$$

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_raw_data, stratified_train_val_test_split
from src.preprocessing.cleaner import deduplicate_and_invalidate, CreditRiskImputer
from src.features.woe_iv import WoEIVAnalyzer
from src.utils import load_config

cfg = load_config()
df_interim, _ = deduplicate_and_invalidate(load_raw_data())
df_train, df_val, df_test = stratified_train_val_test_split(df_interim, random_state=42)

df_train_imp = CreditRiskImputer("grade_median").fit(df_train).transform(df_train)
woe_analyzer = WoEIVAnalyzer(num_bins=10, laplace_alpha=0.5).fit(
    df_train_imp,
    cfg["data_schema"]["numeric_columns"],
    cfg["data_schema"]["categorical_columns"],
)

print("Information Value (IV) Feature Ranking (Training Split, n = 22,691):")
display(woe_analyzer.iv_summary_)

Information Value (IV) Feature Ranking (Training Split, n = 22,691):


,feature,type,num_bins,iv,predictive_power,rank
0,loan_percent_income,Numeric,10,0.8761,Very Strong / Check Policy Variable (> 0.50),1
1,loan_grade,Categorical,7,0.8583,Very Strong / Check Policy Variable (> 0.50),2
2,loan_int_rate,Numeric,10,0.7038,Very Strong / Check Policy Variable (> 0.50),3
3,person_income,Numeric,10,0.4557,Strong Predictor (0.30 - 0.50),4
4,person_home_ownership,Categorical,4,0.3801,Strong Predictor (0.30 - 0.50),5
5,cb_person_default_on_file,Categorical,2,0.1607,Medium Predictor (0.10 - 0.30),6
6,loan_intent,Categorical,6,0.0953,Weak Predictor (0.02 - 0.10),7
7,loan_amnt,Numeric,10,0.0900,Weak Predictor (0.02 - 0.10),8
8,person_emp_length,Numeric,9,0.0597,Weak Predictor (0.02 - 0.10),9
9,person_age,Numeric,10,0.0110,Not Predictive (< 0.02),10


## 2. Detailed Bin-Level WoE & IV Tables for Top Predictors

In [2]:
for feat in ["loan_percent_income", "loan_grade", "loan_int_rate", "person_income", "person_home_ownership", "cb_person_default_on_file"]:
    print(f"\n=== WoE / IV Table: {feat} (Total IV = {woe_analyzer.bin_tables_[feat]['iv_component'].sum():.4f}) ===")
    display(woe_analyzer.bin_tables_[feat].round(4))


=== WoE / IV Table: loan_percent_income (Total IV = 0.8761) ===


,bin,total,events,non_events,event_rate_pct,share_pct,dist_event,dist_non_event,woe,iv_component,feature
0,"(-inf, 0.05]",2375,234,2141,9.85,10.47,0.0472,0.1208,-0.9391,0.0691,loan_percent_income
1,"(0.05, 0.08]",2846,364,2482,12.79,12.54,0.0734,0.1400,-0.6458,0.0430,loan_percent_income
2,"(0.08, 0.1]",2025,255,1770,12.59,8.92,0.0514,0.0998,-0.6631,0.0321,loan_percent_income
3,"(0.1, 0.12]",1869,219,1650,11.72,8.24,0.0442,0.0931,-0.7448,0.0364,loan_percent_income
4,"(0.12, 0.15]",2741,353,2388,12.88,12.08,0.0712,0.1347,-0.6379,0.0405,loan_percent_income
5,"(0.15, 0.18]",2338,428,1910,18.31,10.30,0.0863,0.1077,-0.2222,0.0048,loan_percent_income
6,"(0.18, 0.21]",2045,375,1670,18.34,9.01,0.0756,0.0942,-0.2200,0.0041,loan_percent_income
7,"(0.21, 0.25]",2033,399,1634,19.63,8.96,0.0804,0.0922,-0.1362,0.0016,loan_percent_income
8,"(0.25, 0.32]",2229,787,1442,35.31,9.82,0.1585,0.0813,0.6674,0.0515,loan_percent_income
9,"(0.32, inf]",2190,1548,642,70.68,9.65,0.3118,0.0362,2.1523,0.5930,loan_percent_income



=== WoE / IV Table: loan_grade (Total IV = 0.8583) ===


,bin,total,events,non_events,event_rate_pct,share_pct,dist_event,dist_non_event,woe,iv_component,feature
0,A,7497,760,6737,10.14,33.04,0.1532,0.3800,-0.9086,0.2061,loan_grade
1,B,7271,1189,6082,16.35,32.04,0.2396,0.3430,-0.3590,0.0371,loan_grade
2,C,4528,940,3588,20.76,19.96,0.1894,0.2024,-0.0662,0.0009,loan_grade
3,D,2506,1474,1032,58.82,11.04,0.2969,0.0582,1.6292,0.3889,loan_grade
4,E,674,432,242,64.09,2.97,0.0871,0.0137,1.8515,0.1359,loan_grade
5,F,173,126,47,72.83,0.76,0.0255,0.0027,2.2524,0.0513,loan_grade
6,G,42,41,1,97.62,0.19,0.0084,0.0001,4.5931,0.0380,loan_grade



=== WoE / IV Table: loan_int_rate (Total IV = 0.7038) ===


,bin,total,events,non_events,event_rate_pct,share_pct,dist_event,dist_non_event,woe,iv_component,feature
0,"(-inf, 6.92]",2326,194,2132,8.34,10.25,0.0392,0.1202,-1.1220,0.0910,loan_int_rate
1,"(10.36, 10.99]",2650,454,2196,17.13,11.68,0.0915,0.1239,-0.3028,0.0098,loan_int_rate
2,"(10.99, 11.83]",1836,313,1523,17.05,8.09,0.0631,0.0859,-0.3083,0.0070,loan_int_rate
3,"(11.83, 12.98]",2274,424,1850,18.65,10.02,0.0855,0.1043,-0.1996,0.0038,loan_int_rate
4,"(12.98, 13.85]",2307,510,1797,22.11,10.17,0.1028,0.1014,0.0139,0.0000,loan_int_rate
5,"(13.85, 15.31]",2439,1006,1433,41.25,10.75,0.2026,0.0808,0.9190,0.1119,loan_int_rate
6,"(15.31, inf]",2050,1242,808,60.59,9.03,0.2502,0.0456,1.7024,0.3482,loan_int_rate
7,"(6.92, 7.51]",2526,255,2271,10.10,11.13,0.0514,0.1281,-0.9123,0.0699,loan_int_rate
8,"(7.51, 8.88]",1990,227,1763,11.41,8.77,0.0458,0.0994,-0.7752,0.0416,loan_int_rate
9,"(8.88, 10.36]",2293,337,1956,14.70,10.11,0.0679,0.1103,-0.4847,0.0205,loan_int_rate


... (2 outputs left out)


=== WoE / IV Table: person_home_ownership (Total IV = 0.3801) ===


,bin,total,events,non_events,event_rate_pct,share_pct,dist_event,dist_non_event,woe,iv_component,feature
0,OWN,1782,128,1654,7.18,7.85,0.0259,0.0933,-1.2822,0.0865,person_home_ownership
1,MORTGAGE,9402,1190,8212,12.66,41.43,0.2398,0.4632,-0.6582,0.1470,person_home_ownership
2,RENT,11429,3619,7810,31.67,50.37,0.7291,0.4405,0.5040,0.1455,person_home_ownership
3,OTHER,78,25,53,32.05,0.34,0.0051,0.0030,0.5321,0.0011,person_home_ownership



=== WoE / IV Table: cb_person_default_on_file (Total IV = 0.1607) ===


,bin,total,events,non_events,event_rate_pct,share_pct,dist_event,dist_non_event,woe,iv_component,feature
0,N,18682,3451,15231,18.47,82.33,0.6954,0.8591,-0.2113,0.0346,cb_person_default_on_file
1,Y,4009,1511,2498,37.69,17.67,0.3046,0.1409,0.7707,0.1261,cb_person_default_on_file
